# Salaire ou Compa-Ratio : que faut-il expliquer ?

On part de `employes.xlsx`, `bands.xlsx` et `market.xlsx`.

1. Un seul tableau `df`, avec les ratios
2. Voir les valeurs aberrantes
3. Les traiter
4. Vérifier l'effet, avant et après
5. Les deux matrices de corrélation
6. Comparer les modèles

Le salaire suit en général la grille de près, mais deux salariés au même `Mid` peuvent gagner très différemment. On cherche à expliquer cet écart autour du `Mid`.

Sous chaque graphe, un **Constat** donne les chiffres du fichier chargé. Il est recalculé à chaque lancement. Le texte au-dessus dit seulement comment lire.

Dans le même dossier que ce notebook : les trois fichiers Excel et `outils_eda.py` (la lecture des fichiers et le dessin des graphes). Si les Excel n'y sont pas, ce sont ceux de `input/` du projet qui sont lus ; la première cellule dit lesquels.

Une fois lancé sur les vraies données, ce fichier contient des salaires et des matricules : ne pas l'envoyer sur GitHub sans avoir effacé les sorties.

---
## 1. Un seul tableau

Doublons de matricule retirés, puis chaque salarié reçoit la grille et le marché de son métier et de son niveau.

| Colonne | Calcul | Lecture |
|---|---|---|
| `Compa_Ratio` | salaire / `Mid` | 0,90 = 10 % sous le milieu de grille |
| `Range_Penetration` | (salaire − `Min`) / (`Max` − `Min`) | 0 = au Min, 1 = au Max |
| `Market_Ratio` | salaire / `Median` | 1,00 = payé comme le marché |
| `Ratio_Grille_Marche` | `Mid` / `Median` | 1,10 = la grille paie 10 % de plus que le marché |
| `Amplitude_Grille` | (`Max` − `Min`) / `Mid` | la largeur de la grille |
| `Position_Mid` | (`Mid` − `Min`) / (`Max` − `Min`) | 0,5 = `Mid` pile au milieu |

Les trois premières contiennent le salaire. Les trois suivantes décrivent seulement la grille.

In [ ]:
import numpy as np
import pandas as pd

import outils_eda as outils

# employes.xlsx, bands.xlsx et market.xlsx sont lus dans le dossier du notebook, puis croisés :
# une ligne par salarié, avec Min, Mid, Max et Median de son métier + niveau
df = outils.lire_et_croiser()

df['Age_Embauche'] = df['Age'] - df['Anciennete']
# Les trois mesures de la place du salarié (elles contiennent le salaire).
# outils.diviser laisse la case vide quand le dénominateur est vide, nul ou négatif.
df['Compa_Ratio'] = outils.diviser(df['Fixe_Annuel_MAD'], df['Mid'])
df['Range_Penetration'] = outils.diviser(df['Fixe_Annuel_MAD'] - df['Min'], df['Max'] - df['Min'])
df['Market_Ratio'] = outils.diviser(df['Fixe_Annuel_MAD'], df['Median'])
# Les trois descriptions du poste (elles ne contiennent pas le salaire)
df['Ratio_Grille_Marche'] = outils.diviser(df['Mid'], df['Median'])
df['Amplitude_Grille'] = outils.diviser(df['Max'] - df['Min'], df['Mid'])
df['Position_Mid'] = outils.diviser(df['Mid'] - df['Min'], df['Max'] - df['Min'])

outils.constat_depart(df)
df.head()

Graphes interactifs : molette pour zoomer, double-clic pour revenir, souris sur un point pour voir le matricule.

---
## 2. Voir les valeurs aberrantes

Ici on repère seulement, rien n'est retiré.

* **Erreur de données** (rouge) : une valeur impossible.
* **Valeur extrême** (orange) : un outlier de la boîte à moustaches du Compa-Ratio. On regarde le Compa-Ratio parce que sur le salaire brut, les outliers sont juste les niveaux élevés.

In [ ]:
# ----- Réglages -----
AGE_MIN, AGE_MAX = 18, 75       # la plage d'âge jugée possible
AGE_EMBAUCHE_MIN = 18           # l'âge le plus jeune auquel on peut avoir été embauché
COEF_MOUSTACHE = 1.5            # la longueur des moustaches : 1.5 = boîte à moustaches habituelle ; 3 = seulement les cas très nets

# 1. On écrit les six contrôles d'erreur de données. Chaque contrôle : son nom, et les lignes qu'il attrape (True = en erreur).
# Une valeur vide ne déclenche pas ces contrôles (sauf pour le salaire) : « pas d'âge » n'est pas « âge impossible ».
grille_presente = df['Mid'].notna()
controles = {
    'Salaire vide, nul ou négatif': df['Fixe_Annuel_MAD'].isna() | (df['Fixe_Annuel_MAD'] <= 0),
    'Grille incohérente (Min, Mid, Max)': grille_presente & ((df['Min'] <= 0) | (df['Mid'] <= 0) | (df['Max'] <= 0)
                                                           | (df['Min'] > df['Mid']) | (df['Mid'] > df['Max'])),
    'Médiane du marché nulle ou négative': df['Median'] <= 0,
    f'Âge impossible (hors {AGE_MIN}-{AGE_MAX} ans)': (df['Age'] < AGE_MIN) | (df['Age'] > AGE_MAX),
    'Ancienneté négative': df['Anciennete'] < 0,
    f"Âge à l'embauche impossible (moins de {AGE_EMBAUCHE_MIN} ans)": df['Age_Embauche'] < AGE_EMBAUCHE_MIN,
}
# 2. On note dans la colonne « Raison » les contrôles ratés par chaque ligne (une ligne peut en rater plusieurs)
df['Raison'] = ''
for nom, en_erreur in controles.items():
    df.loc[en_erreur, 'Raison'] += nom + ' ; '
df['Raison'] = df['Raison'].str.rstrip(' ;')
# 3. Une ligne qui a au moins une raison est une erreur de données
en_erreur = df['Raison'] != ''

# 4. On calcule la boîte à moustaches du Compa-Ratio : LIMITES = le bout des deux moustaches.
# Sans les lignes en erreur, pour qu'une ligne fausse ne déplace pas la boîte.
compa_sain = df.loc[~en_erreur, 'Compa_Ratio'].dropna()
q1, q3 = compa_sain.quantile(0.25), compa_sain.quantile(0.75)
LIMITES = (q1 - COEF_MOUSTACHE * (q3 - q1), q3 + COEF_MOUSTACHE * (q3 - q1))
# 5. On repère les outliers (les salariés au-delà des moustaches), et on note pourquoi dans « Raison »
trop_bas = ~en_erreur & (df['Compa_Ratio'] < LIMITES[0])
trop_haut = ~en_erreur & (df['Compa_Ratio'] > LIMITES[1])
df.loc[trop_bas, 'Raison'] = f'Compa-Ratio très bas (sous {outils.fr(LIMITES[0])})'
df.loc[trop_haut, 'Raison'] = f'Compa-Ratio très haut (au-dessus de {outils.fr(LIMITES[1])})'

# 6. On donne un statut à chaque ligne (Normal, Valeur extrême, Erreur de données), pour colorier les graphes
df['Statut'] = np.select([en_erreur, trop_bas | trop_haut], [outils.ERREUR, outils.EXTREME], default=outils.NORMAL)

# 7. On affiche les erreurs de données, contrôle par contrôle (un contrôle qui n'a rien repéré n'est pas affiché).
# Les outliers du Compa-Ratio, eux, se lisent sur la boîte à moustaches, plus bas. Rien n'est encore retiré.
bilan = {nom: int(masque.sum()) for nom, masque in controles.items() if masque.sum() > 0}
if bilan:
    display(pd.DataFrame({'Lignes repérées': bilan}))
outils.constat(f"Erreurs de données : {int(en_erreur.sum())} lignes sur {len(df)} ({outils.part(en_erreur.sum(), len(df))}).")

### Le salaire face au Mid

Un point = un salarié. Ligne noire : salaire = `Mid`. Pointillés : les limites. Orange : extrême. Rouge : erreur.

Si les points suivent la ligne noire, la corrélation est forte. S'ils s'étalent en hauteur pour un même `Mid`, le `Mid` seul ne suffit pas à deviner le salaire.

In [ ]:
outils.graphe_salaire_mid(df, LIMITES)

### La répartition du Compa-Ratio

Nombre de salariés pour chaque valeur du Compa-Ratio. Les barres au-delà des pointillés sont les extrêmes.

In [ ]:
outils.graphe_repartition(df, LIMITES)

### Les autres colonnes

Seulement pour regarder : combien de valeurs sortent des moustaches. Un âge élevé ou une longue ancienneté est un profil rare, on n'y touche pas.

In [ ]:
outils.tableau_extremes(df)

---
## 3. Traiter

* Les erreurs de données sortent toujours de `df`.
* Les extrêmes du Compa-Ratio sortent si `TRAITER_EXTREMES = True`.

Aucun salaire n'est modifié. Les lignes sorties vont dans `ecartes`, avec leur raison.

Les extrêmes du bas sont les cas que la détection cherche. On les retire ici pour apprendre la pratique habituelle, et `ecartes` reste la liste à regarder de près.

In [ ]:
TRAITER_EXTREMES = True   # True : les extrêmes du Compa-Ratio sont mis de côté ; False : ils restent dans df

# 1. On garde une copie du tableau d'avant, pour la comparaison de l'étape 4
df_avant = df.copy()

# 2. On choisit les lignes à mettre de côté : toujours les erreurs de données, et les valeurs extrêmes si le réglage le demande
a_ecarter = (df['Statut'] == outils.ERREUR) | ((df['Statut'] == outils.EXTREME) & TRAITER_EXTREMES)

# 3. On range ces lignes dans « ecartes », avec leur raison : c'est la liste à relire
ecartes = df[a_ecarter].copy()

# 4. On les retire de df. Aucun salaire n'est modifié : df garde les autres lignes telles quelles.
df = df[~a_ecarter].copy()

# 5. On affiche le compte : combien au départ, combien mises de côté, combien gardées
print(f"{len(df_avant)} salariés au départ -> {len(ecartes)} mis de côté -> {len(df)} gardés dans df pour la suite.")

# 6. On affiche combien de lignes sont mises de côté pour chaque raison
display(ecartes['Raison'].value_counts().rename('Lignes mises de côté').to_frame())

# 7. On affiche ces lignes, de la plus éloignée du milieu de grille à la moins éloignée
print("Les lignes mises de côté :")
colonnes_trace = [c for c in ['Matricule', 'Job_Family', 'Niveau', 'Fixe_Annuel_MAD', 'Min', 'Mid', 'Max', 'Compa_Ratio', 'Age', 'Anciennete', 'Raison'] if c in ecartes.columns]
display(ecartes.assign(Ecart=(ecartes['Compa_Ratio'] - 1).abs()).sort_values('Ecart', ascending=False)[colonnes_trace])

---
## 4. Vérifier l'effet

Les mêmes graphes, avant et après, sur les mêmes échelles. Ce qu'on attend :

* plus aucun point en dehors des pointillés ;
* un écart-type du Compa-Ratio plus petit, et une asymétrie plus proche de 0 ;
* une médiane presque inchangée. Si elle bouge, on a retiré trop de monde : augmenter `COEF_MOUSTACHE`.

In [ ]:
# 1. On redessine les deux graphes de l'étape 2, avant et après
outils.graphes_avant_apres(df_avant, df, LIMITES)

# 2. On calcule les mêmes chiffres avant et après (les erreurs de données ne comptent pas dans « avant » : leurs ratios sont faux)
comparaison = pd.DataFrame({'Avant': outils.chiffres_compa(df_avant[df_avant['Statut'] != outils.ERREUR]), 'Après': outils.chiffres_compa(df)})
comparaison['Différence'] = comparaison['Après'] - comparaison['Avant']
display(comparaison.round(2))
outils.constat_avant_apres(comparaison, len(ecartes), len(df_avant))

# 3. On retire de df les deux colonnes de repérage (Statut, Raison) : elles ne servent plus
df = df.drop(columns=['Statut', 'Raison'])

---
## 5. Les deux matrices de corrélation

Une corrélation va de −1 à +1. Rouge (+1) : les deux colonnes montent ensemble. Bleu (−1) : l'une monte quand l'autre descend. Blanc (0) : pas de lien.

Le 9-Box écrit en mots devient un rang (6 = Hauts Potentiels, 1 = Décisions requises). Ses catégories ne sont pas vraiment ordonnées, donc sa ligne donne une tendance.

### Matrice 1 : le salaire

Celle du comité. `Niveau`, `Min`, `Mid`, `Max` et `Median` vont en général ensemble, et le salaire avec eux. Si c'est le cas ici, les autres facteurs paraîtront faibles, écrasés par l'effet du niveau.

In [ ]:
# df_num : une copie pour les matrices, où le 9-Box écrit en mots devient un rang. df, lui, garde le 9-Box en mots.
df_num = outils.en_nombres(df)

COLONNES_MATRICE_1 = ['Fixe_Annuel_MAD', 'Niveau', 'Mid', 'Min', 'Max', 'Median',
                      'Ratio_Grille_Marche', 'Age', 'Age_Embauche', 'Anciennete',
                      'Competence_N1', 'Positionnement_9BOX', 'Hot_job']
outils.matrice(df_num, COLONNES_MATRICE_1, 'Matrice 1 : ce qui va avec le salaire (Fixe_Annuel_MAD)')
outils.constat_liens(df_num, 'Fixe_Annuel_MAD', COLONNES_MATRICE_1, repetitions=True)

### Matrice 2 : le Compa-Ratio

Sans le salaire, ni `Min` et `Max`. La première ligne dit ce qui joue à grille égale.

Cases liées par la formule, à ne pas interpréter :

* `Compa_Ratio`, `Range_Penetration` et `Market_Ratio` entre eux : les trois contiennent le salaire ;
* `Compa_Ratio` avec `Mid`, `Market_Ratio` avec `Median` : le dénominateur ;
* `Market_Ratio` avec `Ratio_Grille_Marche`.

`Range_Penetration` et `Market_Ratio` servent à comparer. Ils ne vont pas dans le modèle.

In [ ]:
# D'abord le Compa-Ratio (ce qu'on cherche à expliquer), puis les deux autres mesures de la place du salarié, puis les facteurs
AUTRES_MESURES = ['Range_Penetration', 'Market_Ratio']
FACTEURS = ['Niveau', 'Mid', 'Median', 'Ratio_Grille_Marche', 'Amplitude_Grille', 'Position_Mid', 'Age', 'Age_Embauche', 'Anciennete',
            'Competence_N1', 'Positionnement_9BOX', 'Hot_job']
outils.matrice(df_num, ['Compa_Ratio'] + AUTRES_MESURES + FACTEURS, 'Matrice 2 : ce qui va avec la place dans la grille (Compa_Ratio)')
# Le Mid est laissé hors du constat : son lien avec le Compa-Ratio vient de la formule
outils.constat_liens(df_num, 'Compa_Ratio', FACTEURS, a_ignorer=['Mid'])

---
## 6. Comparer les modèles

| Modèle | Explique | Avec |
|---|---|---|
| M1 | le salaire | une régression |
| M2 | le salaire | des arbres |
| M3 | le Compa-Ratio | une régression |
| M4 | le Compa-Ratio | des arbres |
| Réf. | le log du salaire | une régression (le modèle actuel du projet) |

Pour M3 et M4, salaire prédit = `Mid` × Compa-Ratio prédit. Tous sont donc jugés sur le salaire.

Chaque salarié est prédit par un modèle qui ne l'a jamais vu : 5 paquets, on apprend sur 4, on prédit le 5ᵉ.

Les colonnes données aux modèles sont dans `COLONNES_MODELE`. Pour en retirer une, effacer sa ligne.

In [ ]:
# Ce que lisent les modèles. « catégorie » : chaque valeur a son propre effet. « nombre » : un seul effet par unité en plus.
COLONNES_MODELE = {
    'Job_Family': 'catégorie',           # le métier
    'Niveau': 'catégorie',               # le niveau
    'Entite_N1': 'catégorie',            # le pôle
    'Mid': 'nombre',                     # le milieu de la grille
    'Ratio_Grille_Marche': 'nombre',     # Mid / médiane du marché
    'Amplitude_Grille': 'nombre',        # la largeur de la grille
    'Position_Mid': 'nombre',            # la place du Mid entre le Min et le Max
    'Anciennete': 'nombre',              # en années
    'Anciennete_Carre': 'nombre',        # l'ancienneté au carré : laisse l'effet de l'ancienneté ralentir avec les années
    'Age_Embauche': 'nombre',            # l'âge à l'arrivée
    'Competence_N1': 'nombre',           # la note de compétence de l'an dernier
    'Performance_N1': 'catégorie',       # la performance de l'an dernier
    'Positionnement_9BOX': 'catégorie',  # la case 9-Box
    'Hot_job': 'nombre',                 # la tension du métier
    'Niveau_Etude_Max': 'catégorie',     # le niveau d'études
    'Cat_Ecole_Max': 'catégorie',        # la catégorie d'école
}

# Les salariés utilisables : il faut un salaire et un Mid (sans Mid, pas de Compa-Ratio)
base = df.dropna(subset=['Fixe_Annuel_MAD', 'Mid', 'Compa_Ratio']).copy()
base['Anciennete_Carre'] = base['Anciennete'] ** 2
print(f"Salariés utilisés pour les modèles : {len(base)} sur {len(df)} (les autres n'ont pas de grille).")

X = outils.tableau_modele(base, COLONNES_MODELE)
mid, salaire, compa_ratio = base['Mid'].values, base['Fixe_Annuel_MAD'].values, base['Compa_Ratio'].values

predictions = {
    'M1 : salaire, régression': outils.predire('regression', X, salaire),
    'M2 : salaire, arbres': outils.predire('arbres', X, salaire),
    'M3 : Compa-Ratio, régression': mid * outils.predire('regression', X, compa_ratio),
    'M4 : Compa-Ratio, arbres': mid * outils.predire('arbres', X, compa_ratio),
    # Le modèle actuel du projet : logarithme du salaire, avec le Mid lui aussi en logarithme
    'Réf. : log du salaire, régression (modèle actuel)': np.exp(outils.predire('regression', X.assign(Mid=np.log(X['Mid'])), np.log(salaire))),
}

resultats = outils.juger(predictions, salaire, mid)
display(resultats.round(2))
outils.constat_modeles(predictions, resultats, salaire)

* **MAE** : l'erreur moyenne, dans l'unité du salaire.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **Erreur médiane en %** : pour un salarié sur deux, le modèle tombe à moins de ce %.
* **R² sur le salaire** : haut dès que le salaire suit la grille, même si le modèle n'apprend rien d'autre.
* **R² sur le Compa-Ratio** : la part de l'écart autour du `Mid` que le modèle explique. C'est le chiffre à regarder. S'il est faible, il manque une information dans les colonnes (historique des augmentations, date de recrutement…).

Si les arbres ne font pas nettement mieux, garder la régression : elle s'explique colonne par colonne.